# Phase 1: Legal Language Model Pre-training - Legal-CapsuleRoute

## Task
Train a legal language model from scratch. Given a 512-word prompt from a legal document, the model writes the next 128 words. Scored with BERTScore F1 (main) and ROUGE-L F1.

Author: Preet Patel (261110043)

Model: no attention. Each token vector is split into capsule groups, and information flows from nearby earlier tokens by routing-by-agreement (an iterative process with the capsule "squash" function). Training uses mixed precision (fp16 + GradScaler).

## Input
- Phase-1 legal text from the Hugging Face dataset Exploration-Lab/CS-781-Capstone (needs the HF_TOKEN secret).
- Official Phase-1 test prompts from the same dataset.

## Flow of the notebook (from input to output)
1. Load all helper functions (memory helpers, tokenizer, data download, training, generation, scoring).
2. Set the training settings and create the output folders.
3. Log in and download the legal documents, removing very noisy ones.
4. Load the official test prompts.
5. Train a byte-level BPE tokenizer from scratch on the legal text.
6. Build the CapsuleRoute model and check its parameter count.
7. Split the documents randomly into 90% train and 10% validation.
8. Run a quick speed test, then train the model and score it on validation prompts (BERTScore, ROUGE-L).
9. Save the best model weights and a score record.
10. Generate continuations for the test prompts and package them.

## Output
- Best model checkpoint (.pt) and score record (.json).
- submission.zip with the test continuations.

In [11]:
# Import libraries and define all helper functions
import os, json, gc, ctypes, random, zipfile, resource, subprocess, sys, time, math, re as _re
from pathlib import Path
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

try:
    from tqdm.auto import tqdm
except Exception:
    # It takes it (items) and **kw (extra options) as input, returns the same items; it is the fallback progress bar that just returns the items unchanged.
    def tqdm(it, **kw):
        return it

torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

# It takes text as input, gives a document a noise score from 0 to 1 (higher means messier text) and returns a number from 0 (clean) to 1 (noisy).
def doc_noise_score(text):
    if not text:
        return 1.0
    n = len(text)
    standard = sum(1 for c in text if c.isalnum() or c.isspace() or c in ".,;:\'\"()-/%&")
    nonstandard_ratio = 1 - (standard / n)

    run_chars, i = 0, 0
    while i < n:
        j = i
        while j < n and text[j] == text[i]:
            j += 1
        if j - i >= 8:
            run_chars += (j - i)
        i = j
    run_ratio = run_chars / n

    words = text.split()
    if words:
        bad = sum(1 for w in words if len(w) > 25 or not any(c.isalpha() for c in w))
        fragment_ratio = bad / len(words)
    else:
        fragment_ratio = 1.0

    return min(1.0, (nonstandard_ratio + run_ratio + fragment_ratio) / 3)

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        t0 = time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

# It takes k (step number) and msg (message) as input, prints which step of the notebook is running and returns nothing.
def stage(k, msg):
    print(f"\n>> capsuleroute step {k}/10: {msg}")

# It returns how much RAM this process is using (GB).
def ram_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)

# It takes tag (label) as input, prints the current RAM usage with a label and returns nothing.
def log_ram(tag):
    print(f"   ram[{tag}]={ram_gb():.2f}GB")

# It frees unused CPU and GPU memory and returns nothing.
def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

# It takes net (model) as input, uses all available GPUs if there is more than one and returns the model, wrapped for several GPUs if available.
def multi_gpu(net):
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"   multi-GPU: {torch.cuda.device_count()} GPUs -> nn.DataParallel")
        return nn.DataParallel(net)
    return net

# It takes net (model) as input, gets the plain model back from the multi-GPU wrapper and returns the plain model.
def unwrap(net):
    return net.module if isinstance(net, nn.DataParallel) else net

NOTEBOOK_CLOCK = time.time()

# It returns how many hours have passed since the start.
def hours_elapsed():
    return (time.time() - NOTEBOOK_CLOCK) / 3600.0

# It takes ema_state (moving-average weights), net (model) and decay (decay factor) as input, updates the moving-average copy of the model weights and returns nothing.
def ema_update_(ema_state, net, decay):
    with torch.no_grad():
        for k, v in unwrap(net).state_dict().items():
            if k in ema_state and v.dtype.is_floating_point:
                ema_state[k].mul_(decay).add_(v.detach().float(), alpha=1.0 - decay)

# It takes path (file path), net (model), opt (optimizer), scaler (gradient scaler), step, ema_state (moving-average weights), best_val (best validation loss), best_state (best weights) and loss_history (loss history) as input, saves everything needed to resume training later and returns nothing.
def save_train_checkpoint(path, net, opt, scaler, step, ema_state, best_val, best_state, loss_history):
    torch.save({
        "step": step, "model": unwrap(net).state_dict(), "opt": opt.state_dict(),
        "scaler": scaler.state_dict(), "ema": ema_state, "best_val": best_val,
        "best_state": best_state, "loss_history": loss_history,
    }, path)

# It takes path (file path), net (model), opt (optimizer), scaler (gradient scaler) and dev (device) as input, loads a saved training state to resume training and returns the loaded checkpoint dictionary.
def load_train_checkpoint(path, net, opt, scaler, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    unwrap(net).load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    return ck

# It takes step_logits (scores for the next token), temperature, top_k (top-k) and top_p (top-p) as input, picks the next token using temperature, top-k and top-p sampling and returns the chosen token id for each row.
def sample_next_token(step_logits, temperature, top_k, top_p):
    logits = step_logits.float()
    if not temperature or temperature <= 0:
        return logits.argmax(dim=-1)
    logits = logits / temperature
    if top_k and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    probs = F.softmax(logits, dim=-1)
    if top_p and top_p < 1.0:
        sorted_probs, sorted_idx = probs.sort(dim=-1, descending=True)
        keep = (sorted_probs.cumsum(dim=-1) - sorted_probs) <= top_p
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        choice = torch.multinomial(sorted_probs, 1)
        return sorted_idx.gather(-1, choice).squeeze(-1)
    return torch.multinomial(probs, 1).squeeze(-1)

# It makes sure the ijson library is installed and returns the ijson module.
def ensure_ijson():
    global ijson
    try:
        import ijson
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ijson"])
        import ijson
    return ijson

# It finds the Hugging Face login token and returns the token string.
def resolve_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t:
            print("auth: Kaggle Secrets HF_TOKEN")
            return t
    except Exception:
        pass
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            print("auth: Colab Secrets HF_TOKEN")
            return t
    except Exception:
        pass
    t = os.environ.get("HF_TOKEN", "")
    if t:
        print("auth: HF_TOKEN env var")
        return t
    try:
        import getpass
        return getpass.getpass("Enter your Hugging Face token (hidden; dataset is gated): ")
    except Exception:
        return ""

# It takes url (file URL), token (login token), cap (maximum value), label (name to print), doc_char_cap (max characters per document) and sink (list to fill) as input, downloads a big JSON file piece by piece and keeps up to a limit of documents and returns a number of documents kept (None if it failed).
def stream_capped(url, token, cap, label, doc_char_cap, sink):
    ij = ensure_ijson()
    import requests
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    noise_max = getattr(Cfg, "NOISE_DOC_MAX", 1.0)
    got, rejected_noisy = 0, 0
    try:
        with requests.get(url, headers=headers, stream=True, timeout=180) as resp:
            resp.raise_for_status()
            for item in ij.items(resp.raw, "item"):
                text = item.get("text", "") if isinstance(item, dict) else ""
                if not (text and len(text) > 100):
                    continue
                text = text[:doc_char_cap]
                if doc_noise_score(text) > noise_max:
                    rejected_noisy += 1
                    continue
                sink.append({"text": text, "source": label})
                got += 1
                if got % 200 == 0:
                    free_mem()
                    print(f"   {label}: {got} docs ({rejected_noisy} rejected so far for noise > {noise_max})")
                if got >= cap:
                    break
    except Exception as e:
        print(f"   {label}: direct stream failed ({e})")
        return None
    if rejected_noisy:
        print(f"   {label}: {rejected_noisy} candidate document(s) rejected outright for doc_noise_score > {noise_max}")
    return got

# It takes repo_id (dataset name), source_paths (data file paths), token (login token), per_source_cap (max documents per source), total_cap (max documents in total) and doc_char_cap (max characters per document) as input, downloads and clean the training documents from all sources and returns a list of documents.
def collect_corpus(repo_id, source_paths, token, per_source_cap, total_cap, doc_char_cap):
    from datasets import load_dataset
    docs = []
    any_ok = False
    for name, path in source_paths.items():
        url = f"https://huggingface.co/datasets/{repo_id}/resolve/main/{path}"
        print(f"streaming '{name}' (cap {per_source_cap})...")
        n = stream_capped(url, token, per_source_cap, name, doc_char_cap, docs)
        if n is None:
            print(f"  falling back to datasets library for '{name}'...")
            try:
                ds = load_dataset(repo_id, data_files=path, split="train", streaming=True,
                                   token=token if token else None)
                n = 0
                noise_max = getattr(Cfg, "NOISE_DOC_MAX", 1.0)
                for item in ds:
                    text = item.get("text", "")
                    if not (text and len(text) > 100):
                        continue
                    text = text[:doc_char_cap]
                    if doc_noise_score(text) > noise_max:
                        continue
                    docs.append({"text": text, "source": name})
                    n += 1
                    if n % 200 == 0:
                        free_mem()
                    if n >= per_source_cap:
                        break
            except Exception as e:
                print(f"  fallback also failed: {e}")
                n = 0
        print(f"  -> {n or 0} docs")
        any_ok = any_ok or bool(n)
        if len(docs) >= total_cap:
            break
    if not any_ok:
        print("all per-source loading failed -- falling back to merged 'train' split.")
        merged = load_dataset(repo_id, name="phase_1", split="train", streaming=True,
                               token=token if token else None)
        for item in merged:
            text = item.get("text", "")
            if text and len(text) > 100:
                docs.append({"text": text[:doc_char_cap], "source": "mixed"})
                if len(docs) >= total_cap:
                    break
    docs = docs[:total_cap]
    free_mem()
    return docs

_PROMPT_FIELDS = ["text", "prompt", "input", "input_text", "document", "context", "source"]

# It takes d (one test record) as input, gets the prompt text from one test record and returns prompt text (empty if missing).
def _pull_prompt(d):
    for k in _PROMPT_FIELDS:
        v = d.get(k) if isinstance(d, dict) else None
        if v:
            return v
    return ""

# It takes keys as input, checks whether the record keys are just numbers and returns True or False.
def _looks_flattened(keys):
    if not keys:
        return False
    digit_like = sum(1 for k in keys if str(k).lstrip("-").isdigit())
    return digit_like / len(keys) > 0.9

# It takes repo_id (dataset name) and token (login token) as input, loads the official test prompts and their ids and returns (test ids, prompts).
def collect_test_prompts(repo_id, token):
    from datasets import load_dataset
    ids, prompts = [], []
    try:
        stream = load_dataset(repo_id, name="phase_1", split="test", streaming=True,
                               token=token if token else None)
    except Exception as e:
        print(f"test stream unavailable: {e}")
        return ids, prompts
    try:
        first = next(iter(stream))
    except StopIteration:
        print("test stream produced zero rows")
        return ids, prompts
    if _looks_flattened(first.keys()):
        print("flattened format detected: row keys are IDs, values are prompts")
        for tid, val in first.items():
            p = _pull_prompt(val) if isinstance(val, dict) else (val if isinstance(val, str) else "")
            if p:
                ids.append(str(tid))
                prompts.append(p)
    else:
        try:
            stream = load_dataset(repo_id, name="phase_1", split="test", streaming=True,
                                   token=token if token else None)
        except Exception as e:
            print(f"could not reopen test stream: {e}")
            stream = []
        for item in stream:
            p = _pull_prompt(item)
            tid = str(item.get("id", len(ids)))
            if p:
                ids.append(tid)
                prompts.append(p)
    return ids, prompts

# It takes items (list of items), val_frac (validation fraction) and seed (random seed) as input, randomly splits documents into train and validation and returns (train documents, validation documents, split name).
def split_random(items, val_frac, seed):
    idx = list(range(len(items)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(len(idx) * val_frac))
    held = set(idx[:n_val])
    return [d for i, d in enumerate(items) if i not in held], [d for i, d in enumerate(items) if i in held], "random_90_10"

# It takes items (list of items) and seed (random seed) as input, keeps one whole data source aside for validation and returns (train documents, validation documents, split name).
def split_source_holdout(items, seed):
    counts = Counter(d["source"] for d in items)
    if len(counts) <= 1:
        print("only one source present -- falling back to a random split (seed+1)")
        tr, va, _ = split_random(items, 0.2, seed + 1)
        return tr, va, "random_80_20_seed1"
    holdout = counts.most_common(1)[0][0]
    return ([d for d in items if d["source"] != holdout],
            [d for d in items if d["source"] == holdout], f"source_holdout_{holdout}")

# It takes items (list of items) and val_frac (validation fraction) as input, uses the longest documents as validation and returns (train documents, validation documents, split name).
def split_length_stratified(items, val_frac):
    by_len = sorted(items, key=lambda d: len(d["text"]))
    n_val = max(1, int(len(items) * val_frac))
    return by_len[:-n_val], by_len[-n_val:], "length_stratified"

class SeqDataset(Dataset):
    # It takes items (list of items), tok (tokenizer) and max_len (max length) as input, stores the documents, tokenizer and sequence length and returns nothing.
    def __init__(self, items, tok, max_len):
        self.texts = [d["text"] for d in items]
        self.tok = tok
        self.max_len = max_len

    # It returns the number of documents.
    def __len__(self):
        return len(self.texts)

    # It takes i (index) as input, returns the token ids of one document.
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.max_len, truncation=True,
                        padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0)}

# It takes net (model), val_items (validation documents), tok (tokenizer), cfg (settings), dev (device), use_amp (use mixed precision) and sample_cap (max samples) as input, computes the loss on a sample of validation documents and returns the average validation loss.
@torch.no_grad()
def compute_val_loss_amp(net, val_items, tok, cfg, dev, use_amp, sample_cap=200):
    net.eval()
    ds = SeqDataset(val_items, tok, cfg.SEQ_LEN)
    n = min(sample_cap, len(ds))
    if n == 0:
        net.train()
        return float("nan")
    idx = torch.randperm(len(ds))[:n]
    total_loss, total_count = 0.0, 0
    for start in range(0, n, cfg.BATCH_SIZE):
        chunk_idx = idx[start:start + cfg.BATCH_SIZE]
        full_ids = torch.stack([ds[int(i)]["input_ids"] for i in chunk_idx]).to(dev)
        inputs, targets = full_ids[:, :-1], full_ids[:, 1:]
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = net(inputs)
            loss = F.cross_entropy(logits.reshape(-1, cfg.VOCAB_SIZE), targets.reshape(-1),
                                    ignore_index=tok.pad_token_id, reduction="sum")
        n_real = (targets != tok.pad_token_id).sum().item()
        total_loss += loss.item()
        total_count += max(n_real, 1)
        del logits, loss
    net.train()
    return total_loss / max(total_count, 1)

# It takes net (model), train_items (training documents), tok (tokenizer), cfg (settings), dev (device), total_steps (total steps), val_items (validation documents) and desc (progress label) as input, trains the model with mixed precision and returns (losses, history, best weights).
def train_amp(net, train_items, tok, cfg, dev, total_steps, val_items=None, desc="train"):
    bs = max(1, cfg.BATCH_SIZE)
    resume_path = getattr(cfg, "RESUME_TRAINING_PATH", None)
    while True:
        try:
            return _train_amp_inner(net, train_items, tok, cfg, dev, total_steps, bs, val_items, desc, resume_path)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs <= 1:
                raise
            new_bs = max(1, bs // 2)
            print(f"    CUDA OOM during training at batch_size={bs} -- releasing memory and "
                  f"retrying from scratch at batch_size={new_bs}...")
            del e
            free_mem()
            bs = new_bs

# It takes step, total_steps (total steps), peak_lr (peak learning rate) and cfg (settings) as input, returns learning rate for this step; it is the learning rate at a given step: warm-up first, then slow decay.
def lr_at_step(step, total_steps, peak_lr, cfg):
    warmup_steps = max(1, int(total_steps * cfg.WARMUP_FRAC))
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cos = 0.5 * (1 + math.cos(math.pi * progress))
    floor = cfg.MIN_LR_FRAC * peak_lr
    return floor + (peak_lr - floor) * cos

# It takes net (model), train_items (training documents), tok (tokenizer), cfg (settings), dev (device), total_steps (total steps), batch_size (batch size), val_items (validation documents), desc (progress label) and resume_path (checkpoint to resume) as input, returns (losses, history, best weights); it is the main training loop: batches, loss, updates, validation and saving.
def _train_amp_inner(net, train_items, tok, cfg, dev, total_steps, batch_size, val_items, desc, resume_path=None):
    ds = SeqDataset(train_items, tok, cfg.SEQ_LEN)
    loader_ = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0,
                          pin_memory=torch.cuda.is_available())
    opt = AdamW(net.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    use_amp = dev.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    net.train()
    it = iter(loader_)
    losses, lrs, history = [], [], []
    best_val = float("inf")
    best_state = None
    eval_every = max(1, getattr(cfg, "EVAL_EVERY", total_steps))
    label_smoothing = getattr(cfg, "LABEL_SMOOTHING", 0.0)

    ema_decay = getattr(cfg, "EMA_DECAY", 0.0)
    ema_state = ({k: v.detach().float().clone() for k, v in unwrap(net).state_dict().items()
                  if v.dtype.is_floating_point} if ema_decay > 0 else None)

    start_step = 0
    if resume_path and os.path.exists(resume_path):
        ck = load_train_checkpoint(resume_path, net, opt, scaler, dev)
        start_step = ck["step"]
        if ema_state is not None and ck.get("ema") is not None:
            ema_state = {k: v.to(dev).float() for k, v in ck["ema"].items()}
        best_val = ck["best_val"]
        best_state = ck["best_state"]
        history = ck["loss_history"]
        print(f"   RESUME: loaded training checkpoint {resume_path!r} at step {start_step}/{total_steps}")

    session_hours = getattr(cfg, "SESSION_HOURS", None)
    reserve_hours = getattr(cfg, "RESERVE_HOURS", 0.0)
    time_budget_hours = (session_hours - reserve_hours) if session_hours is not None else None
    ckpt_dir = globals().get("CKPT_DIR", ".")
    arch_slug = globals().get("ARCH_SLUG", "capsuleroute")

    bar = tqdm(range(start_step, total_steps), desc=desc, unit="step", leave=True)
    for step in bar:
        if time_budget_hours is not None and hours_elapsed() >= time_budget_hours:
            resume_ckpt = os.path.join(ckpt_dir, f"{arch_slug}_time_budget_resume.pt")
            save_train_checkpoint(resume_ckpt, net, opt, scaler, step, ema_state, best_val, best_state, history)
            print(f"   TIME BUDGET: {hours_elapsed():.2f}h elapsed >= session_hours-reserve_hours="
                  f"{time_budget_hours:.2f}h -- stopping at step {step}/{total_steps} and saving a "
                  f"resumable checkpoint to {resume_ckpt} (set Cfg.RESUME_TRAINING_PATH to that file "
                  f"to finish this run in a later session). This is a wall-clock safety stop, NOT "
                  f"early stopping -- it has nothing to do with validation-loss trend.")
            del opt, scaler
            free_mem()
            return losses, history, best_state
        cur_lr = lr_at_step(step, total_steps, cfg.LR, cfg)
        for g in opt.param_groups:
            g["lr"] = cur_lr
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loader_)
            batch = next(it)
        full_ids = batch["input_ids"].to(dev)
        inputs, targets = full_ids[:, :-1], full_ids[:, 1:]
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = net(inputs)
            loss = F.cross_entropy(logits.reshape(-1, cfg.VOCAB_SIZE), targets.reshape(-1),
                                    ignore_index=tok.pad_token_id, label_smoothing=label_smoothing)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        if ema_state is not None:
            ema_update_(ema_state, net, ema_decay)
        losses.append(loss.item()); lrs.append(cur_lr)
        try:
            bar.set_postfix(loss=f"{loss.item():.3f}", lr=f"{cur_lr:.2e}", amp=use_amp)
        except Exception:
            pass
        del logits, loss
        if (step + 1) % 50 == 0:
            print(f"   [{desc}] step {step+1}/{total_steps}: loss={sum(losses[-50:])/len(losses[-50:]):.4f} "
                  f"lr={cur_lr:.2e}")
            free_mem()

        is_last = (step + 1) == total_steps
        if val_items and ((step + 1) % eval_every == 0 or is_last):
            if ema_state is not None:
                raw_backup = {k: v.detach().clone() for k, v in unwrap(net).state_dict().items()}
                unwrap(net).load_state_dict(
                    {k: v.to(raw_backup[k].dtype) if k in raw_backup else v for k, v in ema_state.items()},
                    strict=False)
                val_loss = compute_val_loss_amp(net, val_items, tok, cfg, dev, use_amp)
                unwrap(net).load_state_dict(raw_backup, strict=False)
            else:
                val_loss = compute_val_loss_amp(net, val_items, tok, cfg, dev, use_amp)
            recent_train = sum(losses[-eval_every:]) / len(losses[-eval_every:])
            improved = val_loss < best_val
            weights_used = "ema" if ema_state is not None else "raw"
            history.append({"step": step + 1, "train_loss": recent_train, "val_loss": val_loss,
                             "weights": weights_used})
            print(f"    [eval] step {step+1}/{total_steps}: train_loss={recent_train:.4f} "
                  f"val_loss={val_loss:.4f} ({weights_used} weights)"
                  f"{'  <-- new best' if improved else ''}")
            if improved:
                best_val = val_loss
                best_state = ({k: v.detach().cpu().clone() for k, v in ema_state.items()} if ema_state is not None
                               else {k: v.detach().cpu().clone() for k, v in unwrap(net).state_dict().items()})
            free_mem()
    del opt, scaler
    free_mem()
    return losses, history, best_state

# It takes net (model), tok (tokenizer), prompts, cfg (settings), dev (device), max_new_tokens (tokens to generate), batch_size (batch size) and desc (progress label) as input, generates text, using smaller batches if the GPU runs out of memory and returns a list of generated texts.
def safe_generate_batched(net, tok, prompts, cfg, dev, max_new_tokens, batch_size=None, desc="generating"):
    bs = max(1, batch_size or cfg.GEN_BATCH)
    while True:
        try:
            return generate_batched(net, tok, prompts, cfg, dev, max_new_tokens, batch_size=bs, desc=desc)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs <= 1:
                raise
            new_bs = max(1, bs // 2)
            print(f"    CUDA OOM at batch_size={bs} ({desc}) -- releasing memory and retrying at "
                  f"batch_size={new_bs}...")
            del e
            free_mem()
            bs = new_bs

# It takes net (model), tok (tokenizer), prompts, cfg (settings), dev (device), max_new_tokens (tokens to generate), batch_size (batch size), desc (progress label), min_chars (minimum characters) and max_retries (max tries) as input, generates again for outputs that came out too short and returns a list of generated texts.
def generate_with_min_length_retry(net, tok, prompts, cfg, dev, max_new_tokens, batch_size=None,
                                    desc="generating", min_chars=None, max_retries=None):
    min_chars = cfg.GEN_MIN_CHARS if min_chars is None else min_chars
    max_retries = cfg.GEN_MIN_LEN_RETRIES if max_retries is None else max_retries
    out = safe_generate_batched(net, tok, prompts, cfg, dev, max_new_tokens, batch_size=batch_size, desc=desc)
    for attempt in range(max_retries):
        short_idx = [i for i, t in enumerate(out) if len(t or "") < min_chars]
        if not short_idx:
            break
        print(f"    {len(short_idx)} prediction(s) under {min_chars} chars -- retry "
              f"{attempt + 1}/{max_retries} with a different seed...")
        torch.manual_seed(cfg.SEED + 1000 + attempt)
        retried = safe_generate_batched(net, tok, [prompts[i] for i in short_idx], cfg, dev, max_new_tokens,
                                         batch_size=batch_size, desc=f"{desc}-retry{attempt + 1}")
        for i, text in zip(short_idx, retried):
            if len(text or "") > len(out[i] or ""):
                out[i] = text
    return out

# It takes net (model), tok (tokenizer), prompts, cfg (settings), dev (device), max_new_tokens (tokens to generate), batch_size (batch size) and desc (progress label) as input, generates continuations for many prompts at once and returns a list of generated texts.
@torch.no_grad()
def generate_batched(net, tok, prompts, cfg, dev, max_new_tokens, batch_size=None, desc="generating"):
    batch_size = batch_size or cfg.GEN_BATCH
    net.eval()
    tok.padding_side = "right"
    out = [None] * len(prompts)
    eos_id, pad_id = tok.eos_token_id, tok.pad_token_id

    for start in tqdm(range(0, len(prompts), batch_size), desc=desc, unit="batch", leave=True):
        chunk = prompts[start:start + batch_size]
        enc = tok(chunk, truncation=True, max_length=cfg.PROMPT_LEN, padding=True, return_tensors="pt")
        cur = enc["input_ids"].to(dev)
        real_len = enc["attention_mask"].sum(dim=1).to(dev)
        b = cur.size(0)
        cur_len = real_len.clone()
        done = torch.zeros(b, dtype=torch.bool, device=dev)
        rows = torch.arange(b, device=dev)

        temperature = getattr(cfg, "GEN_TEMPERATURE", 0.0)
        top_k_sample = getattr(cfg, "GEN_TOP_K", 0)
        top_p_sample = getattr(cfg, "GEN_TOP_P", 1.0)
        for _ in range(max_new_tokens):
            logits = net(cur)
            T = cur.size(1)
            read_pos = (cur_len - 1).clamp(min=0, max=T - 1)
            idx = read_pos.view(b, 1, 1).expand(b, 1, logits.size(-1))
            step_logits = logits.gather(1, idx).squeeze(1)
            next_tok = sample_next_token(step_logits, temperature, top_k_sample, top_p_sample)
            del logits
            pad_col = torch.full((b, 1), pad_id, dtype=cur.dtype, device=dev)
            cur = torch.cat([cur, pad_col], dim=1)
            active = ~done
            old_vals = cur[rows, cur_len]
            cur[rows, cur_len] = torch.where(active, next_tok, old_vals)
            done = done | (active & (next_tok == eos_id))
            cur_len = cur_len + active.long()
            del next_tok

        cur_cpu, cur_len_cpu, real_len_cpu = cur.tolist(), cur_len.tolist(), real_len.tolist()
        for i in range(b):
            row = cur_cpu[i][real_len_cpu[i]:cur_len_cpu[i]]
            if eos_id in row:
                row = row[:row.index(eos_id)]
            out[start + i] = tok.decode(row, skip_special_tokens=True)
        del cur, cur_len, done
        free_mem()
    return out

# It takes items (list of items), tok (tokenizer), prompt_len (prompt length), cont_len (continuation length) and cap (maximum value) as input, makes prompt and reference pairs from validation documents and returns a list of prompt / reference pairs.
def build_references(items, tok, prompt_len, cont_len, cap=None):
    cap = Cfg.DEV_SET_CAP if cap is None else cap
    refs = []
    for d in items:
        ids = tok(d["text"], truncation=True, max_length=prompt_len + cont_len)["input_ids"]
        if len(ids) < prompt_len + 10:
            continue
        refs.append({
            "prompt": tok.decode(ids[:prompt_len], skip_special_tokens=True),
            "reference": tok.decode(ids[prompt_len:prompt_len + cont_len], skip_special_tokens=True),
        })
        if len(refs) >= cap:
            break
    return refs

# It takes hyp (generated text) and ref (reference text) as input, computes the ROUGE-L F1 score between two texts and returns ROUGE-L F1 score.
def rouge_l_f1(hyp, ref):
    h, r = hyp.split(), ref.split()
    if not h or not r:
        return 0.0
    dp = [[0] * (len(r) + 1) for _ in range(len(h) + 1)]
    for i in range(1, len(h) + 1):
        for j in range(1, len(r) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if h[i-1] == r[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    prec, rec = lcs / len(h), lcs / len(r)
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

_bertscore_ok = None

# It checks whether BERTScore can be used and returns True or False.
def bertscore_available():
    global _bertscore_ok
    if _bertscore_ok is not None:
        return _bertscore_ok
    try:
        import bert_score
        _bertscore_ok = True
    except Exception:
        try:
            os.system("pip install --quiet bert-score")
            import bert_score
            _bertscore_ok = True
        except Exception as e:
            print(f"  bert-score unavailable ({e}); using ROUGE-L fallback.")
            _bertscore_ok = False
    return _bertscore_ok

# It takes net (model), val_items (validation documents), tok (tokenizer), cfg (settings) and dev (device) as input, generates on validation prompts and computes the scores and returns a dictionary with BERTScore and ROUGE-L.
def evaluate_split(net, val_items, tok, cfg, dev):
    refs = build_references(val_items, tok, cfg.PROMPT_LEN, cfg.CONT_LEN)
    if not refs:
        return {"n": 0, "bertscore_f1": None, "rouge_l_f1": 0.0}
    prompts = [r["prompt"] for r in refs]
    hyps = safe_generate_batched(net, tok, prompts, cfg, dev, cfg.CONT_LEN,
                                  batch_size=min(cfg.GEN_BATCH, len(prompts)), desc="validating")
    references = [r["reference"] for r in refs]
    rouge = [rouge_l_f1(h, r) for h, r in zip(hyps, references)]
    result = {"n": len(refs), "rouge_l_f1": sum(rouge) / len(rouge)}
    if bertscore_available():
        from bert_score import score as bert_score_fn
        try:
            P, R, F1 = bert_score_fn(hyps, references, lang="en", verbose=False)
            result["bertscore_f1"] = F1.mean().item()
        except Exception as e:
            print(f"  bert-score scoring failed at runtime ({e}); using ROUGE-L only.")
            result["bertscore_f1"] = None
    else:
        result["bertscore_f1"] = None
    return result

stage(1, "utils loaded")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}" + (f" ({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)" if torch.cuda.is_available() else ""))
log_ram("boot")


>> capsuleroute step 1/10: utils loaded
device=cuda (Tesla T4, 15.6 GB)
   ram[boot]=3.72GB


In [12]:
# Set training settings and output folders
stage(2, "config")
WORK_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
OUT_DIR = os.path.join(WORK_DIR, "outputs")
for d in (CKPT_DIR, OUT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

class Cfg:
    SMOKE_TEST = False

    PROMPT_LEN = 512
    CONT_LEN = 128
    SEQ_LEN = PROMPT_LEN + CONT_LEN

    TRAIN_TOKENS_TARGET = 655_361_025
    CHARS_PER_TOKEN_GUESS = 4.8

    VOCAB_SIZE = 50000
    DOC_CHAR_CAP = 6000
    NOISE_DOC_MAX = 0.5
    BPE_TRAIN_SAMPLE_DOCS = 3000
    BPE_TRAIN_CHARS_PER_DOC = 3000
    BATCH_SIZE = 4
    LR = 3e-4
    WARMUP_FRAC = 0.06
    MIN_LR_FRAC = 0.10
    WEIGHT_DECAY = 0.05
    LABEL_SMOOTHING = 0.1
    GRAD_CLIP = 1.0
    SEED = 99
    TRAIN_STEPS = 22000
    EVAL_EVERY = 500

    EMA_DECAY = 0.999

    SESSION_HOURS = 11.5
    RESERVE_HOURS = 0.8
    RESUME_TRAINING_PATH = None

    GEN_BATCH = 32
    GEN_TEMPERATURE = 0.8
    GEN_TOP_K = 50
    GEN_TOP_P = 0.95
    GEN_MIN_CHARS = 20
    GEN_MIN_LEN_RETRIES = 3
    DEV_SET_CAP = 60
    RESUME_CHECKPOINT_PATH = None
    RESUME_TOKENIZER_PATH = None

    _N_SOURCES = 4
    _target_total_chars = int(TRAIN_TOKENS_TARGET * CHARS_PER_TOKEN_GUESS)
    DOCS_PER_SOURCE_CAP = max(50, _target_total_chars // (_N_SOURCES * DOC_CHAR_CAP))
    TOTAL_DOCS_CAP = DOCS_PER_SOURCE_CAP * _N_SOURCES

    if SMOKE_TEST:
        TRAIN_STEPS = 300
        EVAL_EVERY = 100
        _SMOKE_GEN_CAP = 20
        EMA_DECAY = 0.0
        SESSION_HOURS = 999.0
        RESERVE_HOURS = 0.0
        DEV_SET_CAP = 8
        TRAIN_TOKENS_TARGET = TRAIN_STEPS * BATCH_SIZE * SEQ_LEN + 1
        VOCAB_SIZE = 1200
        BPE_TRAIN_SAMPLE_DOCS = 200
        BPE_TRAIN_CHARS_PER_DOC = 800
        _target_total_chars = int(TRAIN_TOKENS_TARGET * CHARS_PER_TOKEN_GUESS)
        DOCS_PER_SOURCE_CAP = max(20, _target_total_chars // (_N_SOURCES * DOC_CHAR_CAP))
        TOTAL_DOCS_CAP = DOCS_PER_SOURCE_CAP * _N_SOURCES

class Config:
    SEQ_LEN = Cfg.SEQ_LEN
    VOCAB_SIZE = Cfg.VOCAB_SIZE

random.seed(Cfg.SEED); np.random.seed(Cfg.SEED); torch.manual_seed(Cfg.SEED)
REPO_ID = "Exploration-Lab/CS-781-Capstone"
SOURCE_PATHS = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}
print(f"sequence length = {Cfg.SEQ_LEN} (prompt {Cfg.PROMPT_LEN} + continuation {Cfg.CONT_LEN})")
print("training strategy: automatic mixed precision (torch.cuda.amp) -- see UTILS/train_amp() for why")
_natural_tokens = Cfg.TRAIN_STEPS * Cfg.BATCH_SIZE * Cfg.SEQ_LEN + 1
print(f"token budget: TRAIN_TOKENS_TARGET={Cfg.TRAIN_TOKENS_TARGET:,} tokens (this config's own "
      f"steps*batch*seq_len+1 product is {_natural_tokens:,} -- TRAIN_TOKENS_TARGET is the explicit "
      f"corpus-sizing cap used regardless of that product) -> ~{Cfg.DOCS_PER_SOURCE_CAP:,} docs/source "
      f"streamed at CHARS_PER_TOKEN_GUESS={Cfg.CHARS_PER_TOKEN_GUESS}")
print("\n" + "=" * 78)
if Cfg.SMOKE_TEST:
    print(f"SMOKE_TEST = True -- running the FULL real pipeline (same corpus-sizing logic, model "
          f"size, batch shape, AMP) at TRAIN_STEPS={Cfg.TRAIN_STEPS} (vs the real 22000) and capped "
          f"official generation ({Cfg._SMOKE_GEN_CAP} prompts), ~30 min total.")
    print("Run this once. If it completes cleanly with no CUDA OOM, set Cfg.SMOKE_TEST = False and "
          "re-run for the real 22000-step training (expected several hours; the SESSION_HOURS/"
          "RESERVE_HOURS wall-clock safety stop will checkpoint and exit cleanly before a Kaggle "
          "session limit kills it, if needed -- see Cfg.RESUME_TRAINING_PATH to continue).")
    print("A clean smoke-test pass is a strong (not 100%) signal the full run will not OOM, since "
          "batch shapes and every memory-relevant setting are identical -- only step COUNT differs.")
else:
    print(f"SMOKE_TEST = False -- running the REAL full configuration: TRAIN_STEPS={Cfg.TRAIN_STEPS}, "
          f"full corpus, full official test set generation. Expected 10-12 hours. The OOM-retry-and-"
          f"halve safety net (UTILS) is the last line of defense if this still hits a CUDA OOM.")
print("=" * 78)


>> capsuleroute step 2/10: config
sequence length = 640 (prompt 512 + continuation 128)
training strategy: automatic mixed precision (torch.cuda.amp) -- see UTILS/train_amp() for why
token budget: TRAIN_TOKENS_TARGET=655,361,025 tokens (this config's own steps*batch*seq_len+1 product is 56,320,001 -- TRAIN_TOKENS_TARGET is the explicit corpus-sizing cap used regardless of that product) -> ~131,072 docs/source streamed at CHARS_PER_TOKEN_GUESS=4.8

SMOKE_TEST = False -- running the REAL full configuration: TRAIN_STEPS=22000, full corpus, full official test set generation. Expected 10-12 hours. The OOM-retry-and-halve safety net (UTILS) is the last line of defense if this still hits a CUDA OOM.


In [ ]:
# Log in to Hugging Face and load the training data
stage(3, "auth + training corpus")
hf_token = resolve_hf_token()
if not hf_token:
    print("WARNING: no HF token found -- the dataset is gated, loading will fail below.")
    print("Get one at https://huggingface.co/settings/tokens and request access to")
    print(f"https://huggingface.co/datasets/{REPO_ID}, then re-run.")
else:
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("auth: authenticated with the Hub")
    except Exception as e:
        print(f"auth warning: {e}")

corpus = collect_corpus(REPO_ID, SOURCE_PATHS, hf_token, Cfg.DOCS_PER_SOURCE_CAP, Cfg.TOTAL_DOCS_CAP, Cfg.DOC_CHAR_CAP)
source_counts = Counter(d["source"] for d in corpus)
print(f"\ncorpus size = {len(corpus)} docs; by source: {dict(source_counts)}")
log_ram("post-corpus")


>> capsuleroute step 3/10: auth + training corpus
auth: Kaggle Secrets HF_TOKEN
auth: authenticated with the Hub
streaming 'acts' (cap 131072)...
   acts: 200 docs (0 rejected so far for noise > 0.5)
   acts: 400 docs (0 rejected so far for noise > 0.5)
   acts: 600 docs (0 rejected so far for noise > 0.5)
   acts: 800 docs (0 rejected so far for noise > 0.5)
   acts: 1000 docs (0 rejected so far for noise > 0.5)
   acts: 1200 docs (0 rejected so far for noise > 0.5)
   acts: 1400 docs (0 rejected so far for noise > 0.5)
   acts: 1600 docs (0 rejected so far for noise > 0.5)
   acts: 1800 docs (0 rejected so far for noise > 0.5)
   acts: 2000 docs (0 rejected so far for noise > 0.5)
   acts: 2200 docs (0 rejected so far for noise > 0.5)
   acts: 2400 docs (0 rejected so far for noise > 0.5)
   acts: 2600 docs (0 rejected so far for noise > 0.5)
   acts: 2800 docs (0 rejected so far for noise > 0.5)
   acts: 3000 docs (0 rejected so far for noise > 0.5)
   acts: 3200 docs (0 rejected s

In [ ]:
# Load the official test prompts
stage(4, "official test prompts")
official_ids, official_prompts = collect_test_prompts(REPO_ID, hf_token)
print(f"collected {len(official_prompts)} test prompts")
log_ram("post-testset")

In [15]:
# Train our own tokenizer on the legal text
stage(5, "tokenizer (from-scratch byte-level BPE on this project's own corpus)")
if getattr(Cfg, "RESUME_TOKENIZER_PATH", None):
    print(f"RESUME MODE: Cfg.RESUME_TOKENIZER_PATH={Cfg.RESUME_TOKENIZER_PATH!r} -- loading a "
          f"previously-trained tokenizer instead of training a new one.")
    tokenizer = LegalByteBPETokenizer.load(Cfg.RESUME_TOKENIZER_PATH)
else:
    _train_sample = random.Random(Cfg.SEED).sample(corpus, min(Cfg.BPE_TRAIN_SAMPLE_DOCS, len(corpus)))
    tokenizer = LegalByteBPETokenizer()
    tokenizer.train([d["text"] for d in _train_sample], target_vocab_size=Cfg.VOCAB_SIZE,
                     sample_chars_per_doc=Cfg.BPE_TRAIN_CHARS_PER_DOC)
    del _train_sample
Cfg.VOCAB_SIZE = len(tokenizer)
Config.VOCAB_SIZE = Cfg.VOCAB_SIZE
print(f"vocab size = {len(tokenizer)}")
print("pretokenization: space-attached (GPT-2-style) -- a leading space attaches to the following "
      "word, so a word usually costs ~1.3 tokens instead of ~2.3 (see _PRETOK_RE in UTILS).")

_measure_sample = corpus[:min(200, len(corpus))]
_measured_chars = sum(len(d["text"]) for d in _measure_sample)
_measured_tokens = sum(len(tokenizer.encode(d["text"])) for d in _measure_sample) or 1
measured_chars_per_token = _measured_chars / _measured_tokens
print(f"measured chars/token on this corpus = {measured_chars_per_token:.3f} "
      f"(guess was {Cfg.CHARS_PER_TOKEN_GUESS})")
del _measure_sample

    BPE training: 49742 merges learned (target 49742) in 188.6s over 48663 unique pretoken types -> final vocab size = 50000
vocab size = 50000
pretokenization: space-attached (GPT-2-style) -- a leading space attaches to the following word, so a word usually costs ~1.3 tokens instead of ~2.3 (see _PRETOK_RE in UTILS).
measured chars/token on this corpus = 4.541 (guess was 4.8)


In [16]:
# Build the CapsuleRoute model
stage(6, "model: Legal-CapsuleRoute (Windowed Dynamic-Routing Capsules)")
ARCH_NAME = "Legal-CapsuleRoute (Windowed Dynamic-Routing Capsules)"
ARCH_SLUG = "legal_capsuleroute_random"

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(dim)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

# It takes s (capsule vectors) and dim (axis) as input, squashes capsule vectors so their length stays between 0 and 1 and returns squashed vectors.
def squash(s, dim=-1):
    norm2 = (s ** 2).sum(dim=dim, keepdim=True)
    scale = norm2 / (1 + norm2)
    return scale * s / torch.sqrt(norm2 + 1e-8)

class CapsuleRoutingLayer(nn.Module):
    # It takes dim (model size), num_caps (number of capsules), window (window size) and routing_iters (routing rounds) as input, creates the capsule routing layer and returns nothing.
    def __init__(self, dim, num_caps=8, window=16, routing_iters=3):
        super().__init__()
        assert dim % num_caps == 0, "dim must be divisible by num_caps"
        self.num_caps = num_caps
        self.cap_dim = dim // num_caps
        self.window = window
        self.routing_iters = routing_iters
        self.transform = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    # It takes x (input tensor) as input, routes information between capsules over nearby tokens and returns output vectors.
    def forward(self, x):
        B, T, D = x.shape
        W = self.window
        x_pad = F.pad(x, (0, 0, W - 1, 0))
        neigh = x_pad.unfold(1, W, 1).permute(0, 1, 3, 2)

        votes = self.transform(neigh)
        votes = votes.view(B, T, W, self.num_caps, self.cap_dim)

        b = torch.zeros(B, T, W, self.num_caps, device=x.device, dtype=x.dtype)
        v = None
        for _ in range(self.routing_iters):
            c = torch.softmax(b, dim=2)
            s = (c.unsqueeze(-1) * votes).sum(2)
            v = squash(s, dim=-1)
            agreement = (votes * v.unsqueeze(2)).sum(-1)
            b = b + agreement

        out = v.reshape(B, T, D)
        return self.norm(x + out)

class LegalCapsuleRouteLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), num_caps (number of capsules), window (window size), routing_iters (routing rounds), max_len (max length) and dropout (dropout rate) as input, builds the full CapsuleRoute language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, num_caps=8, window=16, routing_iters=3, max_len=1024, dropout=0.1):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(CapsuleRoutingLayer(dim, num_caps, window, routing_iters))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    # It takes input_ids (token ids) as input, runs the model and returns next-token scores.
    def forward(self, input_ids):
        x = self.drop(self.embed(input_ids))
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.norm_f(x))

# It creates a new model with the chosen settings and returns a new model.
def build_model():
    return LegalCapsuleRouteLM(Config.VOCAB_SIZE, dim=512, n_layers=6, num_caps=8, window=16,
                                routing_iters=3, max_len=Config.SEQ_LEN + 8)

_probe_net = build_model()
_n_params = sum(p.numel() for p in _probe_net.parameters())
print(f"{ARCH_NAME}: {_n_params:,} parameters")
if not Cfg.SMOKE_TEST:
    assert 50_000_000 <= _n_params <= 300_000_000, "parameter count outside 50M-300M!"
else:
    print("  (SMOKE_TEST=True -- skipping the 50M-300M floor check; the real run's full-size "
          "VOCAB_SIZE target will bring this back into range)")
del _probe_net


>> capsuleroute step 6/10: model: Legal-CapsuleRoute (Windowed Dynamic-Routing Capsules)
Legal-CapsuleRoute (Windowed Dynamic-Routing Capsules): 65,769,296 parameters


In [17]:
# Split data into train and validation (90/10)
stage(7, "split: Random 90/10 holdout")
SPLIT_KEY = "random"
if SPLIT_KEY == "random":
    train_items, val_items, split_name = split_random(corpus, 0.1, Cfg.SEED)
elif SPLIT_KEY == "source_holdout":
    train_items, val_items, split_name = split_source_holdout(corpus, Cfg.SEED)
else:
    train_items, val_items, split_name = split_length_stratified(corpus, 0.1)
print(f"split={split_name}: train={len(train_items)} val={len(val_items)}")


>> capsuleroute step 7/10: split: Random 90/10 holdout
split=random_90_10: train=302815 val=33646


In [18]:
# Quick speed test, then train and evaluate the model
stage(8, "benchmark")
probe_docs = corpus[:max(Cfg.BATCH_SIZE * 4, 8)]
if len(probe_docs) >= Cfg.BATCH_SIZE:
    probe_net = multi_gpu(build_model().to(device))
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_amp(probe_net, probe_docs, tokenizer, Cfg, device, total_steps=5, desc="probe")
    sec_per_step = (time.time() - t0) / 5
    print(f"{sec_per_step:.2f} sec/step ({device}); {Cfg.TRAIN_STEPS} steps -> ~{Cfg.TRAIN_STEPS*sec_per_step/60:.1f} min")
    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"peak GPU memory: {peak_gb:.2f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    if official_prompts:
        sample = official_prompts[:min(Cfg.GEN_BATCH, len(official_prompts))]
        t0 = time.time()
        safe_generate_batched(probe_net, tokenizer, sample, Cfg, device, Cfg.CONT_LEN, batch_size=len(sample), desc="probe-gen")
        sec_per_ex = (time.time() - t0) / len(sample)
        print(f"{sec_per_ex:.2f} sec/example generating -> ~{sec_per_ex*len(official_prompts)/60:.1f} min for full test set")
    del probe_net
    free_mem()
else:
    print("not enough docs collected to benchmark yet -- skipping.")

stage(9, f"training on split={split_name}")
torch.manual_seed(Cfg.SEED)

if getattr(Cfg, "RESUME_CHECKPOINT_PATH", None):
    model = multi_gpu(build_model().to(device))
    unwrap(model).load_state_dict(torch.load(Cfg.RESUME_CHECKPOINT_PATH, map_location=device))
    train_losses, loss_history, best_state = [0.0], [], None
    print(f"\nTRAINING SKIPPED: Cfg.RESUME_CHECKPOINT_PATH={Cfg.RESUME_CHECKPOINT_PATH!r} is set.")
else:
    model = multi_gpu(build_model().to(device))
    train_losses, loss_history, best_state = train_amp(
        model, train_items, tokenizer, Cfg, device, Cfg.TRAIN_STEPS, val_items=val_items, desc=SPLIT_KEY)
    print(f"training ran the full {Cfg.TRAIN_STEPS} steps to completion "
          f"({len(loss_history)} validation checks along the way).")

val_metrics = evaluate_split(model, val_items, tokenizer, Cfg, device)
val_metrics["final_train_loss"] = train_losses[-1]
val_metrics["avg_train_loss"] = sum(train_losses) / len(train_losses)
score_str = (f"bertscore_f1={val_metrics['bertscore_f1']:.4f}" if val_metrics["bertscore_f1"] is not None
             else f"rouge_l_f1={val_metrics['rouge_l_f1']:.4f} (bert-score unavailable)")
print(f"\n-> final_train_loss={val_metrics['final_train_loss']:.4f}  val_n={val_metrics['n']}  {score_str}")

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters = {n_params:,} (must be within 50M-300M)")
if not Cfg.SMOKE_TEST:
    assert 50_000_000 <= n_params <= 300_000_000, "parameter count outside 50M-300M!"
else:
    print("  (SMOKE_TEST=True -- skipping the 50M-300M floor check)")
del model
free_mem()


>> capsuleroute step 8/10: benchmark
   multi-GPU: 2 GPUs -> nn.DataParallel


/tmp/ipykernel_48/2585344554.py:691: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


probe:   0%|          | 0/5 [00:00<?, ?step/s]

/tmp/ipykernel_48/2585344554.py:755: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


0.32 sec/step (cuda); 22000 steps -> ~116.1 min
peak GPU memory: 3.53 / 15.6 GB


probe-gen:   0%|          | 0/1 [00:00<?, ?batch/s]

... (5 outputs left out)

validating:   0%|          | 0/2 [00:00<?, ?batch/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



-> final_train_loss=2.9972  val_n=60  bertscore_f1=0.8126
parameters = 65,769,296 (must be within 50M-300M)


In [19]:
# Save the best model and its score
stage(9, "save checkpoint + score record")
metric_used = "bertscore_f1" if val_metrics["bertscore_f1"] is not None else "rouge_l_f1"
metric_value = val_metrics["bertscore_f1"] if val_metrics["bertscore_f1"] is not None else val_metrics["rouge_l_f1"]

ckpt_path_root = os.path.join(WORK_DIR, f"{ARCH_SLUG}_best_{split_name}.pt")
torch.save(best_state, ckpt_path_root)
torch.save(best_state, os.path.join(CKPT_DIR, f"{ARCH_SLUG}_best_{split_name}.pt"))
print(f"saved: {ckpt_path_root}")

score_record = {
    "architecture": ARCH_NAME, "arch_slug": ARCH_SLUG, "split_key": SPLIT_KEY,
    "split_name": split_name, "metric_name": metric_used, "metric_value": metric_value,
    "final_train_loss": val_metrics["final_train_loss"], "val_n": val_metrics["n"],
    "train_n": len(train_items),
}
score_path = os.path.join(WORK_DIR, f"score_{ARCH_SLUG}.json")
with open(score_path, "w", encoding="utf-8") as f:
    json.dump(score_record, f, indent=2)
print(f"score record: {score_path}")
print(f"\nValidation {metric_used} for this split is saved in the score record; the matching "
      f"checkpoint is used for the test predictions.")


>> capsuleroute step 9/10: save checkpoint + score record
saved: /kaggle/working/legal_capsuleroute_random_best_random_90_10.pt
score record: /kaggle/working/score_legal_capsuleroute_random.json

Validation bertscore_f1 for this split is saved in the score record; the matching checkpoint is used for the test predictions.


In [20]:
# Generate test predictions and make submission zip
stage(10, "generate test predictions + package submission")
best_model = multi_gpu(build_model().to(device))
unwrap(best_model).load_state_dict(best_state)
free_mem()

predictions = {}
if official_prompts:
    gen_ids, gen_prompts = official_ids, official_prompts
    if getattr(Cfg, "SMOKE_TEST", False):
        cap = getattr(Cfg, "_SMOKE_GEN_CAP", len(official_prompts))
        gen_ids, gen_prompts = official_ids[:cap], official_prompts[:cap]
        print(f"SMOKE_TEST=True -- generating only {len(gen_prompts)}/{len(official_prompts)} "
              f"official test prompts (set Cfg.SMOKE_TEST=False for the real full-coverage run).")
    continuations = generate_with_min_length_retry(best_model, tokenizer, gen_prompts, Cfg, device,
                                                    Cfg.CONT_LEN, batch_size=Cfg.GEN_BATCH, desc="official test set")
    for tid, cont in zip(gen_ids, continuations):
        predictions[str(tid)] = {"text": cont}
    print(f"generated {len(predictions)} predictions "
          f"(expected: every official test ID, doc 6.4 -- unless SMOKE_TEST capped it above)")
else:
    print("no test prompts available -- cannot generate predictions.")

if predictions:
    pred_path_root = os.path.join(WORK_DIR, "prediction.json")
    with open(pred_path_root, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)
    pred_path = os.path.join(OUT_DIR, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    zip_path_root = os.path.join(WORK_DIR, f"submission_{ARCH_SLUG}.zip")
    with zipfile.ZipFile(zip_path_root, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path_root, arcname="prediction.json")
    with zipfile.ZipFile(os.path.join(OUT_DIR, f"submission_{ARCH_SLUG}.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print(f"submission: {zip_path_root}  (upload to Codabench ONLY if this split wins)")

    with zipfile.ZipFile(zip_path_root) as zf:
        zip_names = zf.namelist()
        assert zip_names == ["prediction.json"],             f"submission zip must contain only prediction.json at its root, got {zip_names}"
        _reparsed = json.loads(zf.read("prediction.json"))
        assert set(_reparsed) == set(predictions),             "re-parsed zip contents don't match the predictions that were written"
    print(f"zip verified: {zip_path_root} -- root file {zip_names[0]!r}, "
          f"{len(_reparsed)} prediction(s) re-parsed successfully")

print("\n" + "=" * 78)
print(f"SUMMARY -- {ARCH_NAME} / split={SPLIT_KEY}")
print("=" * 78)
print(f"trained split       : {split_name}")
print(f"parameters          : {n_params:,}")
print(f"corpus              : {len(corpus)} docs, by source {dict(source_counts)}")
print(f"local {metric_used:<15}: {metric_value:.4f}")
print(f"test coverage       : {len(predictions)} / {len(official_prompts)}")
print("\nDone. The test predictions are packaged in submission.zip and the "
      "validation score is saved in the score record "
      "(see above).")


>> capsuleroute step 10/10: generate test predictions + package submission
   multi-GPU: 2 GPUs -> nn.DataParallel


official test set:   0%|          | 0/32 [00:00<?, ?batch/s]

generated 1000 predictions (expected: every official test ID, doc 6.4 -- unless SMOKE_TEST capped it above)
submission: /kaggle/working/submission_legal_capsuleroute_random.zip  (upload to Codabench ONLY if this split wins)
zip verified: /kaggle/working/submission_legal_capsuleroute_random.zip -- root file 'prediction.json', 1000 prediction(s) re-parsed successfully

SUMMARY -- Legal-CapsuleRoute (Windowed Dynamic-Routing Capsules) / split=random
trained split       : random_90_10
parameters          : 65,769,296
corpus              : 336461 docs, by source {'acts': 18820, 'high_courts': 131072, 'supreme_court': 55497, 'tribunals': 131072}
local bertscore_f1   : 0.8126
test coverage       : 1000 / 1000

Done. The test predictions are packaged in submission.zip and the validation score is saved in the score record (see above).
